# Transformer-Based Conversational Intelligence System

**A Transformer/NLP pipeline for automated customer-support conversation triage**

This notebook builds an end-to-end system that reads a raw customer-support message and
extracts structured intelligence from it: **intent**, **sentiment**, **urgency/priority**,
**named entities & topics**, and a **short summary** — using fine-tuned and pretrained
Transformer models (DistilBERT, RoBERTa, BERT-NER, BART), not a wrapper around a hosted LLM API.

**Author:** *[Your Name]* &nbsp;|&nbsp; **Date:** *[Run Date]* &nbsp;|&nbsp; **Runtime:** Google Colab (GPU recommended: `Runtime > Change runtime type > T4 GPU`)

---

### Contents
1. Project Overview & Business Problem
2. System Architecture
3. Dataset Selection & Justification
4. Data Loading
5. Exploratory Data Analysis
6. Data Cleaning & Preprocessing
7. Train / Validation / Test Split
8. Transformer Tokenization
9. Intent Classification Model
10. Model Training
11. Model Evaluation
12. Sentiment Analysis
13. Urgency / Priority Detection
14. Entity / Topic Extraction
15. Conversation Summarization
16. End-to-End Conversational Intelligence Pipeline
17. Example Predictions
18. Error Analysis
19. Limitations & Future Improvements
20. Final Results & Conclusion
21. CV Bullet Points

> **A note on honesty of results:** every metric in this notebook is produced by code that
> actually trains/evaluates on real data. Nothing is hard-coded. Where a component has no
> suitable public benchmark (urgency detection, sentiment on this specific domain), that
> gap is called out explicitly, a documented weak-supervision or spot-check strategy is
> used instead, and its limitations are stated plainly rather than dressed up as a formal benchmark.


In [ ]:
# Install dependencies (Colab-friendly, lightweight models throughout)
!pip install -q -U transformers datasets evaluate accelerate sentencepiece pyarrow \
    keybert sentence-transformers scikit-learn seqeval


In [ ]:
import os
import re
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import seaborn as sns

import torch
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, f1_score
)

# ---- Reproducibility -------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIPE_DEVICE = 0 if DEVICE == "cuda" else -1
print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: No GPU detected. Training will still run but will be considerably "
          "slower. In Colab: Runtime > Change runtime type > T4 GPU.")


## 1. Project Overview & Business Problem

Customer-support teams (in fintech, travel/booking, e-commerce, telecom, etc.) receive a
high volume of free-text messages that currently require a human to read, classify, and
prioritize before any action is taken. This creates three concrete business problems:

1. **Slow triage** — urgent, high-risk messages (fraud, cancellations, angry customers)
   can sit in a queue behind routine questions.
2. **Inconsistent routing** — without structured intent labels, messages are often routed
   to the wrong team or skill group.
3. **No aggregate visibility** — support leads cannot easily see *what* customers are
   contacting about, *how* they feel, or *which* topics/entities recur, because the raw
   data is unstructured text.

**Goal of this project.** Build a **Transformer-based Conversational Intelligence System**
that converts one raw customer message into a structured record:

```
Input:  "I've been trying to contact the owner for three days but nobody is
         responding. I want to cancel my booking."

Output: Intent    : Booking/Cancellation-related issue
        Sentiment : Negative
        Urgency   : High
        Entities  : ["three days" (date), "owner" (topic), "booking" (topic)]
        Summary   : "Customer wants to cancel because the owner is unresponsive."
```

This is intentionally built as a **multi-model NLP pipeline** — each capability is powered
by an actual Transformer model that is either **fine-tuned from scratch** on labeled data
(intent, urgency) or **applied via transfer learning** from a pretrained checkpoint
(sentiment, NER, summarization) — rather than a single prompt sent to a hosted LLM API.
That is the point of the exercise: demonstrate the underlying NLP/ML engineering, not just
API orchestration.


## 2. System Architecture

The system is a **branching pipeline**: a single cleaned & tokenized input feeds four
independent classification/extraction heads, whose outputs are combined with an
abstractive summarizer into one structured JSON record.

| Stage | Technique | Model |
|---|---|---|
| Preprocessing | Regex-based cleaning, whitespace/URL normalization | — |
| Tokenization | WordPiece subword tokenization | `distilbert-base-uncased` tokenizer |
| Intent Classification | Supervised fine-tuning (transfer learning) | DistilBERT, fine-tuned |
| Sentiment Analysis | Pretrained transfer learning (zero-shot on this domain) | RoBERTa (Twitter-sentiment) |
| Urgency / Priority | Weak supervision + fine-tuning | DistilBERT, fine-tuned on heuristic labels |
| Entities / Topics | Pretrained NER + embedding-based keyphrase extraction + regex | BERT-NER + KeyBERT (MiniLM) |
| Summarization | Pretrained sequence-to-sequence generation | DistilBART fine-tuned on SAMSum dialogues |

The diagram below is generated directly from code (not a static image) to keep it easy to
edit as the pipeline evolves.


In [ ]:
def draw_architecture_diagram():
    fig, ax = plt.subplots(figsize=(13, 11))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 13)
    ax.axis("off")

    def box(x, y, w, h, text, color="#4C72B0", fontsize=10, textcolor="white"):
        patch = FancyBboxPatch(
            (x, y), w, h,
            boxstyle="round,pad=0.08,rounding_size=0.12",
            linewidth=1.2, edgecolor="#2b2b2b", facecolor=color
        )
        ax.add_patch(patch)
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center",
                 fontsize=fontsize, color=textcolor, wrap=True, weight="bold")
        return (x + w / 2, y, x + w / 2, y + h)  # bottom-center, top-center points

    def arrow(x1, y1, x2, y2):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                     arrowprops=dict(arrowstyle="-|>", color="#333333", lw=1.6))

    # Stage 1: raw input
    _, top1 = box(3.0, 12.0, 4.0, 0.8, "Raw Customer Conversation", color="#55606e")[2:]
    b1_bottom = (5.0, 12.0)

    # Stage 2: cleaning
    b2_top = box(3.0, 10.7, 4.0, 0.8, "Text Cleaning & Preprocessing", color="#55606e")
    arrow(5.0, 12.0, 5.0, 11.5)

    # Stage 3: tokenizer
    box(3.0, 9.4, 4.0, 0.8, "Transformer Tokenizer\n(DistilBERT WordPiece)", color="#55606e")
    arrow(5.0, 10.7, 5.0, 10.2)

    # connector down to branch point
    arrow(5.0, 9.4, 5.0, 8.85)
    ax.plot([1.2, 8.8], [8.85, 8.85], color="#333333", lw=1.6)

    # Stage 4: four parallel heads
    heads = [
        (0.3, "Intent\nClassification\n(DistilBERT,\nfine-tuned)", "#c44e52"),
        (2.85, "Sentiment\nAnalysis\n(RoBERTa,\npretrained)", "#dd8452"),
        (5.4, "Urgency /\nPriority\n(DistilBERT,\nweak-supervised)", "#937860"),
        (7.95, "Entities / Topics\n(BERT-NER +\nKeyBERT + regex)", "#8172b2"),
    ]
    head_tops = []
    for x, label, color in heads:
        ax.plot([x + 0.85, x + 0.85], [8.85, 8.35], color="#333333", lw=1.6)
        box(x, 7.5, 1.9, 0.85, label, color=color, fontsize=8.3)
        head_tops.append((x + 0.95, 7.5))

    # converge into summarizer
    ax.plot([1.2, 8.8], [7.1, 7.1], color="#333333", lw=1.6)
    for x, _ in head_tops:
        ax.plot([x, x], [7.5, 7.1], color="#333333", lw=1.6)
    arrow(5.0, 7.1, 5.0, 6.55)

    # Stage 5: summarization
    box(3.0, 5.7, 4.0, 0.85, "Transformer Summarization\n(DistilBART fine-tuned on SAMSum)", color="#55606e", fontsize=9)
    arrow(5.0, 5.7, 5.0, 5.2)

    # Stage 6: structured output
    box(2.4, 3.9, 5.2, 1.0, "Structured Customer Intelligence Output\n{intent, sentiment, urgency, entities, topics, summary}",
        color="#2ca02c", fontsize=9.5)

    ax.set_title("End-to-End Conversational Intelligence Pipeline", fontsize=14, weight="bold", pad=15)
    plt.tight_layout()
    plt.show()

draw_architecture_diagram()


## 3. Dataset Selection & Justification

Each pipeline component needs a different kind of supervision, and no single dataset
covers all five tasks. The choices below are made explicitly, with the trade-offs stated.

**Intent Classification — [`PolyAI/banking77`](https://huggingface.co/datasets/PolyAI/banking77).**
A widely-used, real customer-service benchmark: ~13k single-turn customer utterances sent
to an online bank's support team, labeled with 77 fine-grained intents
(`card_arrival`, `cancel_transfer`, `lost_or_stolen_card`, `refund_not_showing_up`, ...).
It is domain-realistic (it *is* customer support text), has an official train/test split
(no leakage risk), and is small enough to fine-tune DistilBERT on in minutes on a free
Colab GPU. This is the dataset the intent classifier is genuinely fine-tuned and evaluated on.

**Sentiment Analysis — pretrained `cardiffnlp/twitter-roberta-base-sentiment-latest`.**
Banking77 has no sentiment labels, and no public dataset combines *this exact domain*
with sentiment annotations at a scope suitable for this project. Rather than fabricate
labels, sentiment is produced via **transfer learning from a pretrained 3-class
(negative/neutral/positive) RoBERTa sentiment model**, applied zero-shot to support text.
This is validated qualitatively and against a small, transparently-labeled spot-check set
(Section 12) — not claimed as a rigorously benchmarked number.

**Urgency / Priority — no suitable public dataset exists**, so a **documented weak-supervision
strategy** is used instead (Section 13): a transparent, rule-based heuristic (urgency
keywords, repeated-contact phrases, time expressions, punctuation/caps intensity) assigns
Low/Medium/High labels to Banking77 utterances, and a DistilBERT classifier is fine-tuned
to generalize *beyond* the exact keyword rules. Its held-out evaluation is reported against
both (a) a heuristic-labeled test split and (b) a small hand-written, manually-judged gold
set — with the difference between the two explicitly discussed as a limitation.

**Entities / Topics — pretrained `dslim/bert-base-NER`** (fine-tuned BERT on CoNLL-2003 for
PERSON/ORG/LOC/MISC) **+ KeyBERT** (`all-MiniLM-L6-v2` sentence-embeddings) for
domain keyphrases that generic NER won't catch (e.g. "booking", "refund", "card payment"),
**+ lightweight regex** for money amounts and date/time expressions. No fine-tuning
dataset is needed here — this is a deliberate transfer-learning + classical-NLP combination.

**Summarization — pretrained `philschmid/distilbart-cnn-12-6-samsum`.** This is a
DistilBART checkpoint already fine-tuned on **SAMSum**, a dialogue-summarization dataset —
i.e. it is trained specifically to summarize conversational text, which is a much closer
domain match for customer-support messages than a news-summarization model
(e.g. CNN/DailyMail) would be, while remaining small enough to run on a free GPU.

**Why not just fine-tune everything?** Fine-tuning requires labeled data for the *exact*
task. Two of the five components here (sentiment on this domain, urgency) simply have no
appropriate public labels — using pretrained transfer learning and honestly-scoped weak
supervision for those, while genuinely fine-tuning where labels exist (intent, urgency),
is the methodologically correct choice and is exactly what real-world NLP engineering
looks like.


## 4. Data Loading

Loading the official Banking77 train/test split from the Hugging Face Hub. The test split
is held out untouched until final evaluation (Section 11) — it is never used for training,
model selection, or hyperparameter tuning, which is how data leakage is avoided.

**Note on loading method.** Banking77 (and a number of older Hugging Face dataset repos)
ships with a legacy *loading script* rather than plain data files. Depending on the exact
versions of `datasets` / `huggingface_hub` installed at runtime, that legacy script path
can raise an `HfUriError` unrelated to your data or code — it's a known upstream
incompatibility between old script-based dataset repos and newer Hub client versions. To
sidestep it entirely, the loader below pulls the data straight from Hugging Face's public
**Parquet auto-conversion** (`datasets-server` API), which every dataset on the Hub gets
automatically and which involves no loading script at all. `datasets.load_dataset` is kept
only as a fallback.


In [ ]:
import requests
import pandas as pd

DATASET_ID = "PolyAI/banking77"

# Canonical 77 intent label names, in the same order as the dataset's ClassLabel
# feature (id -> name). Hardcoded as a robust fallback / cross-check since the
# label-name metadata isn't always reachable (see notes below).
BANKING77_LABEL_NAMES = [
    "activate_my_card", "age_limit", "apple_pay_or_google_pay", "atm_support",
    "automatic_top_up", "balance_not_updated_after_bank_transfer",
    "balance_not_updated_after_cheque_or_cash_deposit", "beneficiary_not_allowed",
    "cancel_transfer", "card_about_to_expire", "card_acceptance", "card_arrival",
    "card_delivery_estimate", "card_linking", "card_not_working",
    "card_payment_fee_charged", "card_payment_not_recognised",
    "card_payment_wrong_exchange_rate", "card_swallowed", "cash_withdrawal_charge",
    "cash_withdrawal_not_recognised", "change_pin", "compromised_card",
    "contactless_not_working", "country_support", "declined_card_payment",
    "declined_cash_withdrawal", "declined_transfer",
    "direct_debit_payment_not_recognised", "disposable_card_limits",
    "edit_personal_details", "exchange_charge", "exchange_rate", "exchange_via_app",
    "extra_charge_on_statement", "failed_transfer", "fiat_currency_support",
    "get_disposable_virtual_card", "get_physical_card", "getting_spare_card",
    "getting_virtual_card", "lost_or_stolen_card", "lost_or_stolen_phone",
    "order_physical_card", "passcode_forgotten", "pending_card_payment",
    "pending_cash_withdrawal", "pending_top_up", "pending_transfer", "pin_blocked",
    "receiving_money", "Refund_not_showing_up", "request_refund",
    "reverted_card_payment?", "supported_cards_and_currencies", "terminate_account",
    "top_up_by_bank_transfer_charge", "top_up_by_card_charge",
    "top_up_by_cash_or_cheque", "top_up_failed", "top_up_limits", "top_up_reverted",
    "topping_up_by_card", "transaction_charged_twice", "transfer_fee_charged",
    "transfer_into_account", "transfer_not_received_by_recipient", "transfer_timing",
    "unable_to_verify_identity", "verify_my_identity", "verify_source_of_funds",
    "verify_top_up", "virtual_card_not_working", "visa_or_mastercard",
    "why_verify_identity", "wrong_amount_of_cash_received",
    "wrong_exchange_rate_for_cash_withdrawal",
]


def load_banking77_via_pr_parquet(dataset_id: str = DATASET_ID):
    """Load Banking77 straight from its Parquet files.

    As of this writing, `PolyAI/banking77`'s `main` branch still ships the
    OLD-style Python loading script (`banking77.py`) -- which the current
    `datasets` library refuses to run at all ("Dataset scripts are no longer
    supported"), and which also breaks the `datasets-server` API's automatic
    Parquet conversion for the same reason (ConfigNamesError).
    A pending, not-yet-merged pull request on the repo (currently `refs/pr/7`,
    with `refs/pr/6` and a specific historical commit as older duplicates)
    already deleted the script and added plain Parquet files directly. Hugging
    Face lets you resolve files from an open PR branch the same way as `main`,
    so we pull from there -- no loading script, no `datasets` library version
    sensitivity, no dependency on `datasets-server` at all.
    Once that PR is merged into `main`, the "main" candidate below will start
    working too and will be tried first automatically.
    """
    candidate_revisions = [
        "main",                                        # will work once the PR merges
        "refs/pr/7",                                    # current open PR with parquet files
        "refs/pr/6",                                     # earlier duplicate PR
        "796a4623935746f71378f0ebd435635a8ce08e50",       # pinned commit, extra safety net
    ]
    last_err = None
    for revision in candidate_revisions:
        base = f"https://huggingface.co/datasets/{dataset_id}/resolve/{revision}/data"
        try:
            train_df = pd.read_parquet(f"{base}/train-00000-of-00001.parquet")
            test_df = pd.read_parquet(f"{base}/test-00000-of-00001.parquet")
            return train_df, test_df, revision
        except Exception as e:
            last_err = e
    raise RuntimeError(
        f"Could not fetch Banking77 Parquet files from any known revision "
        f"(tried {candidate_revisions})."
    ) from last_err


def load_banking77_via_datasets_server(dataset_id: str = DATASET_ID):
    """Fallback 1: read the auto-converted Parquet files via the datasets-server API."""
    parquet_resp = requests.get(
        "https://datasets-server.huggingface.co/parquet", params={"dataset": dataset_id}, timeout=30
    )
    parquet_resp.raise_for_status()
    parquet_files = parquet_resp.json()["parquet_files"]

    def url_for(split):
        matches = [f["url"] for f in parquet_files if f["split"] == split]
        if not matches:
            raise ValueError(f"No parquet file found for split '{split}'")
        return matches[0]

    train_df = pd.read_parquet(url_for("train"))
    test_df = pd.read_parquet(url_for("test"))
    return train_df, test_df


def load_banking77_via_datasets_lib(dataset_id: str = DATASET_ID):
    """Fallback 2: the standard `datasets` library loader. Currently expected to fail
    for this dataset until the Parquet-conversion PR is merged into `main`, since the
    library refuses to execute the legacy `banking77.py` loading script."""
    from datasets import load_dataset
    raw = load_dataset(dataset_id)
    train_df = raw["train"].to_pandas()
    test_df = raw["test"].to_pandas()
    label_names = raw["train"].features["label"].names
    return train_df, test_df, label_names


try:
    train_df_full, test_df, used_revision = load_banking77_via_pr_parquet()
    label_names = BANKING77_LABEL_NAMES
    print(f"Loaded Banking77 via direct Parquet files (revision: {used_revision}).")
except Exception as e1:
    print(f"Direct Parquet approach failed ({type(e1).__name__}: {e1}); trying datasets-server...")
    try:
        train_df_full, test_df = load_banking77_via_datasets_server()
        label_names = BANKING77_LABEL_NAMES
        print("Loaded Banking77 via the Hugging Face datasets-server Parquet API.")
    except Exception as e2:
        print(f"datasets-server approach failed ({type(e2).__name__}: {e2}); falling back to `datasets.load_dataset`...")
        train_df_full, test_df, label_names = load_banking77_via_datasets_lib()
        print("Loaded Banking77 via `datasets.load_dataset` fallback.")

NUM_INTENT_LABELS = len(label_names)
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

train_df_full["intent"] = train_df_full["label"].map(id2label)
test_df["intent"] = test_df["label"].map(id2label)

print(f"Train examples : {len(train_df_full)}")
print(f"Test examples  : {len(test_df)}")
print(f"Num intents    : {NUM_INTENT_LABELS}")
train_df_full.head()

## 5. Exploratory Data Analysis

Before modeling, we check class balance and utterance length — both directly inform later
decisions (max sequence length for the tokenizer, whether class weighting is needed).


In [ ]:
plt.figure(figsize=(11, 7))
top_intents = train_df_full["intent"].value_counts().nlargest(20)
sns.barplot(x=top_intents.values, y=top_intents.index, palette="viridis")
plt.title("Top 20 Most Frequent Intents (Training Set)")
plt.xlabel("Number of examples")
plt.ylabel("Intent")
plt.tight_layout()
plt.show()

counts = train_df_full["intent"].value_counts()
print(f"Most frequent intent : {counts.idxmax()}  ({counts.max()} examples)")
print(f"Least frequent intent: {counts.idxmin()}  ({counts.min()} examples)")
print(f"Mean examples/intent : {counts.mean():.1f}   Std: {counts.std():.1f}")


In [ ]:
train_df_full["word_count"] = train_df_full["text"].apply(lambda x: len(x.split()))

plt.figure(figsize=(8, 5))
sns.histplot(train_df_full["word_count"], bins=25, kde=True, color="#4C72B0")
plt.title("Distribution of Utterance Length (Training Set)")
plt.xlabel("Word count")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

print(train_df_full["word_count"].describe())
print(f"\n99th percentile word count: {train_df_full['word_count'].quantile(0.99):.0f}")


**Takeaway:** utterances are short (the 99th percentile is well under 40 words), so a
`max_length` of 64 WordPiece tokens for the tokenizer (Section 8) comfortably covers the
data without excessive padding, keeping training fast. The class distribution is
reasonably balanced across 77 intents but not perfectly uniform — this is accounted for
via stratified splitting (Section 7) and is revisited in the urgency section, where the
weak-label distribution is considerably more skewed and does require explicit class weighting.


## 6. Data Cleaning & Preprocessing

The cleaning here is deliberately **light-touch**: we strip URLs and collapse whitespace,
but we **keep casing and punctuation** (`!`, `?`, `$`, `%`, etc.). Two reasons:

1. Modern subword tokenizers (WordPiece/BPE) handle casing and punctuation natively — there
   is no need to lowercase or strip it for the Transformer models to work well.
2. Punctuation and casing are *signal*, not noise, for sentiment and urgency (e.g. "!!!"
   or ALL CAPS correlate with heightened emotion) — aggressive cleaning would throw away
   information the urgency heuristic and sentiment model rely on.


In [ ]:
def clean_text(text: str) -> str:
    """Light-touch cleaning: normalize whitespace/URLs, keep casing & punctuation."""
    text = str(text).strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text)              # strip URLs
    text = re.sub(r"[^A-Za-z0-9\s\.\,\!\?\'\-\:\$\%]", " ", text)  # drop unusual symbols
    text = re.sub(r"\s+", " ", text).strip()                    # collapse whitespace
    return text

train_df_full["clean_text"] = train_df_full["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

train_df_full[["text", "clean_text"]].sample(5, random_state=SEED)


## 7. Train / Validation / Test Split

The **test set is the official Banking77 test split** — untouched until Section 11. A
**stratified 90/10 split of the official training set** creates the validation set used for
model selection (best checkpoint) during training. Stratifying by label keeps the rare
intents represented in both splits, and using a fixed `SEED` makes the split reproducible.


In [ ]:
train_df, val_df = train_test_split(
    train_df_full,
    test_size=0.1,
    random_state=SEED,
    stratify=train_df_full["label"],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)}   Val: {len(val_df)}   Test: {len(test_df)}")
overlap = set(train_df["text"]) & set(val_df["text"])
print(f"Duplicate utterances shared between train and val: {len(overlap)} (should be 0)")


## 8. Transformer Tokenization

Using the `distilbert-base-uncased` WordPiece tokenizer. DistilBERT is chosen as the
backbone for both fine-tuned classifiers (intent, urgency) because it retains ~97% of
BERT's language understanding at roughly half the parameters and inference time — a
practical choice for a system meant to run on free-tier compute.


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_CHECKPOINT = "distilbert-base-uncased"
MAX_LEN = 64  # justified by the word-count EDA above

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

sample_enc = tokenizer(train_df["clean_text"].iloc[0], truncation=True, max_length=MAX_LEN)
print("Example text  :", train_df["clean_text"].iloc[0])
print("Token ids     :", sample_enc["input_ids"])
print("Tokens        :", tokenizer.convert_ids_to_tokens(sample_enc["input_ids"]))


In [ ]:
def to_hf_dataset(df, text_col="clean_text", label_col="label"):
    """Turn a pandas DataFrame into a tokenized torch-formatted HF Dataset.
    The label column is renamed to 'labels' (plural) because that is the
    keyword argument name `AutoModelForSequenceClassification.forward` expects;
    the Trainer only computes a loss when a 'labels' key is present in the batch.
    """
    d = df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "labels"})
    ds = Dataset.from_pandas(d, preserve_index=False)
    ds = ds.map(
        lambda batch: tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN),
        batched=True,
    )
    ds = ds.remove_columns(["text"])
    ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)

print(train_ds)


## 9. Intent Classification Model

Fine-tuning `distilbert-base-uncased` with a 77-way classification head on top — a
standard transfer-learning setup: the pretrained language representation is adapted to
the intent-classification task via supervised fine-tuning on Banking77.


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed

set_seed(SEED)

intent_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_INTENT_LABELS,
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


## 10. Model Training

Standard fine-tuning hyperparameters for a small Transformer on a ~9-10k example dataset:
a low learning rate (2e-5) to avoid catastrophic forgetting of pretrained knowledge, a few
epochs, and best-checkpoint selection on **validation F1** (not the test set) to avoid any
form of test-set leakage into model selection.


In [ ]:
INTENT_OUTPUT_DIR = "./intent_model_ckpt"

training_args = TrainingArguments(
    output_dir=INTENT_OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=intent_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result.metrics)


## 11. Model Evaluation

Evaluating strictly on the held-out **test set**, which was not used anywhere above — not
for training, not for validation-based checkpoint selection. All metrics below are real
numbers produced by this training run, not illustrative placeholders.


In [ ]:
test_results = trainer.evaluate(test_ds)
print("Test set metrics:")
for k, v in test_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

preds_output = trainer.predict(test_ds)
y_true_intent = preds_output.label_ids
y_pred_intent = np.argmax(preds_output.predictions, axis=-1)

print("\nFull 77-class classification report:\n")
print(classification_report(y_true_intent, y_pred_intent, target_names=label_names, zero_division=0))


In [ ]:
# Confusion matrix restricted to the 15 most frequent intents for readability;
# the full 77-class performance is already captured in the classification_report above.
TOP_N = 15
top_labels_idx = train_df["label"].value_counts().nlargest(TOP_N).index.tolist()

mask = np.isin(y_true_intent, top_labels_idx)
cm = confusion_matrix(y_true_intent[mask], y_pred_intent[mask], labels=top_labels_idx)
cm_labels = [id2label[i] for i in top_labels_idx]

plt.figure(figsize=(12, 10))
sns.heatmap(cm, xticklabels=cm_labels, yticklabels=cm_labels, cmap="Blues", annot=True, fmt="d", cbar=True)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.title(f"Confusion Matrix — Top {TOP_N} Most Frequent Intents (Test Set)")
plt.xlabel("Predicted intent")
plt.ylabel("True intent")
plt.tight_layout()
plt.show()


In [ ]:
# Persist the fine-tuned intent model for the inference pipeline used later in the notebook.
INTENT_MODEL_DIR = "./intent_model_final"
trainer.save_model(INTENT_MODEL_DIR)
tokenizer.save_pretrained(INTENT_MODEL_DIR)

from transformers import pipeline as hf_pipeline

intent_classifier = hf_pipeline(
    "text-classification",
    model=INTENT_MODEL_DIR,
    tokenizer=INTENT_MODEL_DIR,
    device=PIPE_DEVICE,
    top_k=1,
)

def predict_intent(text: str):
    result = intent_classifier(clean_text(text))
    result = result[0] if isinstance(result[0], dict) else result[0][0]
    return result["label"], float(result["score"])

print(predict_intent("I've been trying to contact the owner for three days but nobody is responding. I want to cancel my booking."))


## 12. Sentiment Analysis

As justified in Section 3, Banking77 has no sentiment labels, so sentiment is produced via
**transfer learning from a pretrained checkpoint** (`cardiffnlp/twitter-roberta-base-sentiment-latest`,
a RoBERTa model fine-tuned for 3-class sentiment) applied directly to support text. This is
a legitimate transfer-learning demonstration — the model was never trained on customer-support
language, so its performance here reflects genuine domain-transfer ability.

Because there is no domain-matched benchmark to compute a "real" accuracy against, this
section reports a small, **transparently hand-labeled spot-check set** (18 examples I wrote
and labeled by hand for this notebook, covering clearly positive/neutral/negative support
messages) purely as a sanity check — explicitly **not** presented as a rigorous benchmark.


In [ ]:
sentiment_pipeline = hf_pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=PIPE_DEVICE,
)

print("Model label mapping:", sentiment_pipeline.model.config.id2label)

def predict_sentiment(text: str):
    result = sentiment_pipeline(clean_text(text)[:512])[0]
    return result["label"].capitalize(), float(result["score"])

qualitative_examples = [
    "I've been trying to contact the owner for three days but nobody is responding. I want to cancel my booking.",
    "Thank you so much, the refund was processed super quickly!",
    "Can you tell me how to update my mailing address?",
]
for ex in qualitative_examples:
    label, score = predict_sentiment(ex)
    print(f"[{label} ({score:.2f})]  {ex}")


In [ ]:
# Small, transparently hand-labeled spot-check set (NOT a formal benchmark — see markdown above).
sentiment_spotcheck = [
    ("I've been trying to contact the owner for three days but nobody is responding. I want to cancel my booking.", "Negative"),
    ("This is absolutely unacceptable, I have been charged twice for the same order.", "Negative"),
    ("I am very disappointed with how long this refund is taking.", "Negative"),
    ("Nobody has responded to my emails and I'm getting really frustrated.", "Negative"),
    ("The product arrived broken and customer service has been no help at all.", "Negative"),
    ("I want to cancel my subscription, this service has gone downhill.", "Negative"),
    ("Thank you so much, the refund was processed super quickly!", "Positive"),
    ("Great service as always, really appreciate the quick response.", "Positive"),
    ("I just wanted to say the new update makes everything so much easier, thank you!", "Positive"),
    ("Your support team resolved my issue in minutes, fantastic job.", "Positive"),
    ("I'm really happy with how smoothly the account setup went.", "Positive"),
    ("Thanks for the quick reply, that answers my question perfectly.", "Positive"),
    ("Can you tell me how to update my mailing address?", "Neutral"),
    ("What are your customer service hours?", "Neutral"),
    ("How do I set up a recurring transfer to my savings account?", "Neutral"),
    ("Where can I find my account statements from last year?", "Neutral"),
    ("Is there a way to change the currency displayed in the app?", "Neutral"),
    ("What is the daily withdrawal limit for my account?", "Neutral"),
]

texts, gold_sentiment = zip(*sentiment_spotcheck)
pred_sentiment = [predict_sentiment(t)[0] for t in texts]
sentiment_spotcheck_acc = accuracy_score(gold_sentiment, pred_sentiment)

print(f"Spot-check accuracy: {sentiment_spotcheck_acc:.2%}  (n={len(sentiment_spotcheck)})")
pd.DataFrame({"text": texts, "gold": gold_sentiment, "predicted": pred_sentiment})


## 13. Urgency / Priority Detection

No public dataset labels customer-support text with urgency/priority for this kind of
short, single-turn message. Per the project brief, fabricating labels or metrics is not
an option — instead this section documents a **weak-supervision** approach end-to-end:

1. Write a transparent, inspectable **rule-based heuristic** (keyword/phrase patterns +
   punctuation/caps intensity) that assigns a Low/Medium/High urgency label to any text.
2. Apply it to the (otherwise unlabeled-for-urgency) Banking77 training text to generate
   **weak training labels** — this is standard weak-supervision practice, not fabrication:
   the *labels* are heuristic, and that is stated explicitly everywhere they're used.
3. **Fine-tune** a DistilBERT classifier on these weak labels. The goal is for the model to
   generalize the *underlying pattern* the heuristic captures, rather than merely
   pattern-matching the same keywords (which a regex could already do).
4. Evaluate on **two** held-out sets: (a) a heuristic-labeled test split — this measures how
   well the model reproduces the heuristic on unseen text, and (b) a small, hand-written
   **gold set** with human-judged urgency labels — this checks real-world validity. The gap
   between the two is reported and discussed honestly in the Limitations section.


In [ ]:
HIGH_URGENCY_PATTERNS = [
    r"\bimmediately\b", r"\burgent(ly)?\b", r"\basap\b", r"\bright now\b",
    r"\bemergency\b", r"\bcancel\b", r"\bfraud\b", r"\bstolen\b", r"\bunauthori[sz]ed\b",
    r"\blegal action\b", r"\bcomplaint\b", r"\bnot responding\b", r"\bno response\b",
    r"\bstill waiting\b", r"\bthird time\b", r"\bunacceptable\b", r"\bangry\b",
    r"\bfor \d+ (day|days|week|weeks|hour|hours)\b", r"\bcharged twice\b",
    r"\btoday\b.*\bneed\b", r"!{1,}",
]
MEDIUM_URGENCY_PATTERNS = [
    r"\bwhen will\b", r"\bhow long\b", r"\bplease help\b", r"\bissue\b",
    r"\bproblem\b", r"\bnot working\b", r"\berror\b", r"\bconfused\b",
    r"\bcan someone\b", r"\bwhy (was|is|did)\b",
]

def urgency_heuristic_score(text: str) -> int:
    t = text.lower()
    score = 0
    for pat in HIGH_URGENCY_PATTERNS:
        if re.search(pat, t):
            score += 2
    for pat in MEDIUM_URGENCY_PATTERNS:
        if re.search(pat, t):
            score += 1
    caps_words = re.findall(r"\b[A-Z]{3,}\b", text)  # shouting signal, checked on ORIGINAL casing
    score += len(caps_words)
    return score

def urgency_weak_label(text: str) -> str:
    score = urgency_heuristic_score(text)
    if score >= 3:
        return "High"
    elif score >= 1:
        return "Medium"
    else:
        return "Low"

URGENCY_LABELS = ["Low", "Medium", "High"]
urgency_id2label = {i: l for i, l in enumerate(URGENCY_LABELS)}
urgency_label2id = {l: i for i, l in enumerate(URGENCY_LABELS)}

for df_ in (train_df, val_df, test_df):
    df_["urgency_weak"] = df_["clean_text"].apply(urgency_weak_label)
    df_["urgency_label_id"] = df_["urgency_weak"].map(urgency_label2id)

print("Weak-label distribution (train):")
print(train_df["urgency_weak"].value_counts())


The weak-label distribution above is noticeably imbalanced (most short banking
queries don't contain urgency keywords, so "Low" dominates) — this is expected and is
handled with **class-weighted loss** during fine-tuning rather than by inventing synthetic
examples.


In [ ]:
class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_df["urgency_label_id"].values,
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float)
print("Class weights (Low, Medium, High):", class_weights_np)

urgency_train_ds = to_hf_dataset(train_df, label_col="urgency_label_id")
urgency_val_ds = to_hf_dataset(val_df, label_col="urgency_label_id")
urgency_test_ds = to_hf_dataset(test_df, label_col="urgency_label_id")


In [ ]:
from torch.nn import CrossEntropyLoss

class WeightedTrainer(Trainer):
    """Trainer variant that applies class weights to the cross-entropy loss,
    needed because the weak urgency labels are imbalanced (see above)."""
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

urgency_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(URGENCY_LABELS),
    id2label=urgency_id2label,
    label2id=urgency_label2id,
)

URGENCY_OUTPUT_DIR = "./urgency_model_ckpt"

urgency_training_args = TrainingArguments(
    output_dir=URGENCY_OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    report_to="none",
)

urgency_trainer = WeightedTrainer(
    class_weights=class_weights,
    model=urgency_model,
    args=urgency_training_args,
    train_dataset=urgency_train_ds,
    eval_dataset=urgency_val_ds,
    compute_metrics=compute_metrics,
)

urgency_train_result = urgency_trainer.train()
print(urgency_train_result.metrics)


In [ ]:
# (a) Evaluation against the heuristic-labeled test split.
urgency_test_results = urgency_trainer.evaluate(urgency_test_ds)
urgency_weak_test_f1 = urgency_test_results["eval_f1"]
print("Urgency model vs. WEAK (heuristic) test labels:")
for k, v in urgency_test_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

urgency_preds_output = urgency_trainer.predict(urgency_test_ds)
y_true_urg = urgency_preds_output.label_ids
y_pred_urg = np.argmax(urgency_preds_output.predictions, axis=-1)
print("\n" + classification_report(y_true_urg, y_pred_urg, target_names=URGENCY_LABELS, zero_division=0))

cm_urg = confusion_matrix(y_true_urg, y_pred_urg, labels=[0, 1, 2])
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm_urg, annot=True, fmt="d", cmap="Oranges", xticklabels=URGENCY_LABELS, yticklabels=URGENCY_LABELS)
plt.title("Urgency Confusion Matrix (vs. heuristic test labels)")
plt.xlabel("Predicted")
plt.ylabel("Weak label")
plt.tight_layout()
plt.show()


In [ ]:
# Save the urgency model and build an inference helper before the gold-set check.
URGENCY_MODEL_DIR = "./urgency_model_final"
urgency_trainer.save_model(URGENCY_MODEL_DIR)
tokenizer.save_pretrained(URGENCY_MODEL_DIR)

urgency_classifier = hf_pipeline(
    "text-classification",
    model=URGENCY_MODEL_DIR,
    tokenizer=URGENCY_MODEL_DIR,
    device=PIPE_DEVICE,
    top_k=1,
)

def predict_urgency(text: str):
    result = urgency_classifier(clean_text(text))
    result = result[0] if isinstance(result[0], dict) else result[0][0]
    return result["label"], float(result["score"])


In [ ]:
# (b) Evaluation against a small, hand-written, human-judged GOLD set.
# These 24 examples were written and urgency-labeled by hand for this notebook
# specifically as a real-world sanity check -- they are NOT drawn from Banking77
# and were not used anywhere in training, so this measures genuine generalization
# beyond the heuristic's exact keyword list.
urgency_gold_set = [
    ("I've been trying to contact the owner for three days but nobody is responding. I want to cancel my booking.", "High"),
    ("This is the third time I'm emailing about the same fraud charge on my card and no one has helped me!", "High"),
    ("My card was stolen and someone just used it to make a purchase, please block it immediately.", "High"),
    ("I am extremely angry, I was charged twice for the same transaction and I need this fixed right now.", "High"),
    ("My account has been locked and I have a flight in two hours, I need access immediately.", "High"),
    ("I have already called customer service five times about this refund and nobody calls me back, this is unacceptable.", "High"),
    ("Please cancel my subscription immediately, I was never told about this renewal charge.", "High"),
    ("I need this resolved today, I have already waited a week and I'm losing money because of this.", "High"),
    ("Hi, I noticed an extra charge on my statement, can someone explain what it's for?", "Medium"),
    ("My app keeps showing an error when I try to transfer money, could you help me fix this?", "Medium"),
    ("When will my new card arrive? It's been a few days since I ordered it.", "Medium"),
    ("I'm a bit confused about how the loyalty points work, can you clarify?", "Medium"),
    ("There seems to be a problem with my direct debit setup, can someone look into it?", "Medium"),
    ("I tried resetting my password but it's not working, any idea why?", "Medium"),
    ("Can you tell me why my last payment failed?", "Medium"),
    ("I'd like to know how long the verification process usually takes.", "Medium"),
    ("Can you tell me how to update my mailing address?", "Low"),
    ("What are your customer service hours?", "Low"),
    ("Just wanted to say thanks, the new app update looks great!", "Low"),
    ("How do I set up a recurring transfer to my savings account?", "Low"),
    ("Where can I find my account statements from last year?", "Low"),
    ("Is there a way to change the currency displayed in the app?", "Low"),
    ("Can I add a second card to my account?", "Low"),
    ("What is the daily withdrawal limit for my account?", "Low"),
]

gold_texts, gold_urgency = zip(*urgency_gold_set)
pred_urgency = [predict_urgency(t)[0] for t in gold_texts]
urgency_gold_acc = accuracy_score(gold_urgency, pred_urgency)
urgency_gold_f1 = f1_score(gold_urgency, pred_urgency, average="weighted", zero_division=0)

print(f"Gold spot-check accuracy: {urgency_gold_acc:.2%}   weighted F1: {urgency_gold_f1:.3f}   (n={len(urgency_gold_set)})")
gold_result_df = pd.DataFrame({"text": gold_texts, "gold": gold_urgency, "predicted": pred_urgency})
gold_result_df


## 14. Entity / Topic Extraction

Three complementary techniques, each covering what the others miss:

- **General-purpose NER** (`dslim/bert-base-NER`, a BERT model fine-tuned on CoNLL-2003) —
  catches PERSON / ORGANIZATION / LOCATION / MISC mentions.
- **KeyBERT** (embedding similarity between candidate n-grams and the full text, using the
  `all-MiniLM-L6-v2` sentence-transformer) — surfaces **domain topics** that generic NER
  won't tag, e.g. "cancel booking", "refund", "card payment".
- **Lightweight regex** — pulls out money amounts and date/time expressions
  ("$45.99", "three days", "yesterday"), which are common and highly relevant in support
  messages but are not entity types CoNLL-trained NER models are tuned for.


In [ ]:
from keybert import KeyBERT

ner_pipeline = hf_pipeline(
    "ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=PIPE_DEVICE
)
kw_model = KeyBERT(model="all-MiniLM-L6-v2")

MONEY_RE = re.compile(r"\$\s?\d+(?:\.\d{1,2})?|\b\d+(?:\.\d{1,2})?\s?(?:usd|dollars|eur|euros|gbp|pounds)\b", re.I)
DATE_RE = re.compile(r"\b(\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d+\s+(?:day|days|week|weeks|month|months|hour|hours)\b|yesterday|today|tomorrow)\b", re.I)

def extract_entities_and_topics(text: str, top_k_topics: int = 5):
    ner_ents = ner_pipeline(text)
    named_entities = [
        {"text": e["word"], "type": e["entity_group"], "score": round(float(e["score"]), 3)}
        for e in ner_ents
    ]
    money_matches = [m.group() for m in MONEY_RE.finditer(text)]
    date_matches = [m.group() for m in DATE_RE.finditer(text)]

    keywords = kw_model.extract_keywords(
        text, keyphrase_ngram_range=(1, 2), stop_words="english", top_n=top_k_topics
    )
    topics = [kw for kw, _score in keywords]

    return {
        "named_entities": named_entities,
        "money_mentions": money_matches,
        "date_mentions": date_matches,
        "topics": topics,
    }

demo_text = "I've been trying to contact the owner for three days but nobody is responding. I want to cancel my booking."
extract_entities_and_topics(demo_text)


## 15. Conversation Summarization

Using `philschmid/distilbart-cnn-12-6-samsum` — DistilBART already fine-tuned on **SAMSum**
dialogue summaries, which is a closer domain match to conversational customer messages than
a news-summarization checkpoint. Because the model was fine-tuned on multi-turn chat data
formatted with speaker turns, a single customer utterance is framed as one "Customer:" turn
before being passed in, which keeps it in-distribution for the model.


In [ ]:
summarizer = hf_pipeline(
    "summarization", model="philschmid/distilbart-cnn-12-6-samsum", device=PIPE_DEVICE
)

def format_for_dialogue_summarizer(text: str) -> str:
    return f"Customer: {text.strip()}"

def summarize_conversation(text: str, max_length: int = 40, min_length: int = 8) -> str:
    word_count = len(text.split())
    if word_count < 6:
        return text.strip()  # too short to meaningfully compress further
    formatted = format_for_dialogue_summarizer(text)
    max_len = max(min_length + 4, min(max_length, word_count + 5))
    result = summarizer(formatted, max_length=max_len, min_length=min_length, do_sample=False)
    return result[0]["summary_text"].strip()

for ex in [
    "I've been trying to contact the owner for three days but nobody is responding. I want to cancel my booking.",
    "My card was stolen and someone just used it to make a purchase, please block it immediately and send me a replacement.",
]:
    print(f"INPUT  : {ex}")
    print(f"SUMMARY: {summarize_conversation(ex)}\n")


## 16. End-to-End Conversational Intelligence Pipeline

Combining every component built above into a single inference function,
`analyze_conversation(text)`, that returns one structured record — this is the actual
CV-facing deliverable of the project.


In [ ]:
def analyze_conversation(text: str) -> dict:
    cleaned = clean_text(text)

    intent, intent_conf = predict_intent(cleaned)
    sentiment, sentiment_conf = predict_sentiment(cleaned)
    urgency, urgency_conf = predict_urgency(cleaned)
    ents_topics = extract_entities_and_topics(cleaned)
    summary = summarize_conversation(cleaned)

    return {
        "input_text": text,
        "intent": {"label": intent, "confidence": round(intent_conf, 3)},
        "sentiment": {"label": sentiment, "confidence": round(sentiment_conf, 3)},
        "urgency": {"label": urgency, "confidence": round(urgency_conf, 3)},
        "entities": ents_topics["named_entities"],
        "topics": ents_topics["topics"],
        "money_mentions": ents_topics["money_mentions"],
        "date_mentions": ents_topics["date_mentions"],
        "summary": summary,
    }

def pretty_print_analysis(result: dict) -> None:
    print("=" * 78)
    print(f"INPUT: {result['input_text']}")
    print("-" * 78)
    print(f"Intent      : {result['intent']['label']}  (confidence: {result['intent']['confidence']})")
    print(f"Sentiment   : {result['sentiment']['label']}  (confidence: {result['sentiment']['confidence']})")
    print(f"Urgency     : {result['urgency']['label']}  (confidence: {result['urgency']['confidence']})")
    print(f"Entities    : {result['entities']}")
    print(f"Topics      : {result['topics']}")
    print(f"Money       : {result['money_mentions']}")
    print(f"Dates/Times : {result['date_mentions']}")
    print(f"Summary     : {result['summary']}")
    print("=" * 78)


## 17. Example Predictions

Running the full pipeline on several realistic, varied customer-support messages —
including the exact example from the project brief.


In [ ]:
demo_conversations = [
    "I've been trying to contact the owner for three days but nobody is responding. I want to cancel my booking.",
    "Hi! Just wanted to say thank you, my card arrived earlier than expected and everything works perfectly.",
    "My card was stolen and someone just used it to make a $250 purchase, please block it immediately!!!",
    "Hey, quick question - how do I change the currency shown in the app? Not urgent at all.",
    "I was charged twice for the same transaction on 03/14/2026 and nobody has responded to my last two emails about it.",
    "Can you explain how the loyalty points program works? I signed up last month and I'm a little confused.",
]

for convo in demo_conversations:
    result = analyze_conversation(convo)
    pretty_print_analysis(result)


## 18. Error Analysis

Inspecting where the models actually get it wrong, rather than only reporting aggregate
metrics — this is what separates a real evaluation from a headline number.


In [ ]:
# Intent misclassifications on the test set
misclassified_mask = y_true_intent != y_pred_intent
mis_df = test_df.iloc[np.where(misclassified_mask)[0]].copy()
mis_df["true_intent"] = [id2label[i] for i in y_true_intent[misclassified_mask]]
mis_df["pred_intent"] = [id2label[i] for i in y_pred_intent[misclassified_mask]]

print(f"Total misclassified: {misclassified_mask.sum()} / {len(y_true_intent)} "
      f"({misclassified_mask.mean():.2%})")
mis_df[["text", "true_intent", "pred_intent"]].sample(min(10, len(mis_df)), random_state=SEED)


In [ ]:
# Which intent pairs are confused most often?
confusion_pairs = (
    mis_df.groupby(["true_intent", "pred_intent"]).size().reset_index(name="count")
    .sort_values("count", ascending=False).head(10)
)
confusion_pairs


**Reading the confusions.** Most intent errors in Banking77 fine-tuning are typically
between semantically adjacent intents (e.g. `card_not_working` vs. `card_payment_not_recognised`,
or `cancel_transfer` vs. `beneficiary_not_allowed`) rather than random noise — this is a good
sign, since it shows the model has learned meaningful semantic distinctions and is failing on
genuinely ambiguous phrasing, not failing arbitrarily. Inspect the `confusion_pairs` table
above (populated from your actual run) to see the specific pairs for this fine-tuning run.

**Urgency gold-set errors.** Re-examine `gold_result_df` from Section 13 for rows where
`predicted != gold`. In practice, the most common failure mode for a heuristic-trained
urgency model is **polite but urgent** messages — e.g. a customer who is clearly in a
time-critical situation ("I have a flight in two hours") but phrases it politely with no
angry/urgent keywords. The heuristic has no signal for this, so the fine-tuned model
inherits that blind spot. This is a direct, honest consequence of weak supervision and is
exactly the kind of gap a production system would close with real human-labeled urgency data.


## 19. Limitations & Future Improvements

- **Urgency labels are heuristic-derived, not human-annotated at scale.** The fine-tuned
  urgency model can only be as good as the weak-labeling rules used to train it, plus
  whatever it generalizes beyond them. The 24-example gold set is a useful sanity check,
  not a statistically powered benchmark — a production system would need a properly
  human-labeled dataset (ideally with inter-annotator agreement) of a few thousand examples.
- **Sentiment model is applied zero-shot, off-domain.** It was trained on Twitter text, not
  banking/customer-support language; a fine-tuned sentiment head on domain-labeled data
  would likely outperform it, if such labels were available.
- **Single-turn assumption.** Real support "conversations" are multi-turn with two
  speakers; this pipeline currently treats each input as one customer utterance. Extending
  it to full multi-turn threads would need speaker-aware modeling and longer-context
  summarization.
- **No PII redaction.** A production deployment handling real customer messages would need
  to detect and redact personal data (account numbers, emails, addresses) before any
  logging or downstream storage.
- **Summarizer is not fine-tuned on customer-support dialogue specifically** — SAMSum is
  informal chit-chat between friends, which is a reasonable but imperfect stand-in for
  support conversations. Fine-tuning on a support-specific dialogue-summarization dataset,
  if one were licensed/available, would likely improve summary quality and tone.
- **77-way intent classification has inherent ambiguity** between semantically close
  categories (see Error Analysis) — a hierarchical intent taxonomy (coarse category, then
  fine-grained intent) could reduce this in a real product.
- **Latency/cost**: five separate model forward passes per message is fine for
  asynchronous ticket triage but would need batching/distillation/quantization for
  real-time, high-throughput use.


## 20. Final Results & Conclusion

The cell below prints a consolidated results dashboard pulling together every metric
computed earlier in this notebook, so the real numbers from *your* run are easy to find
in one place (and to quote accurately in Section 21).


In [ ]:
print("FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"Intent Classification — DistilBERT fine-tuned, {NUM_INTENT_LABELS}-class (Banking77 test set)")
print(f"   Accuracy          : {test_results['eval_accuracy']:.4f}")
print(f"   Weighted F1       : {test_results['eval_f1']:.4f}")
print(f"   Weighted Precision: {test_results['eval_precision']:.4f}")
print(f"   Weighted Recall   : {test_results['eval_recall']:.4f}")
print()
print("Urgency Detection — DistilBERT fine-tuned on weak-supervised labels")
print(f"   Weak-label test F1 (vs. heuristic)  : {urgency_weak_test_f1:.4f}")
print(f"   Gold spot-check accuracy (n={len(urgency_gold_set)}): {urgency_gold_acc:.4f}")
print(f"   Gold spot-check weighted F1         : {urgency_gold_f1:.4f}")
print()
print(f"Sentiment — pretrained RoBERTa (zero-shot on this domain)")
print(f"   Spot-check accuracy (n={len(sentiment_spotcheck)}): {sentiment_spotcheck_acc:.4f}")
print("=" * 60)


**Conclusion.** This notebook implemented a full Transformer-based NLP pipeline that
takes a raw customer-support message and produces structured intent, sentiment, urgency,
entity/topic, and summary outputs. Every number reported above comes from an actual trained
or pretrained model evaluated on real (held-out, non-leaked) data; where no suitable labeled
dataset existed (urgency, domain-specific sentiment), that gap was addressed with explicitly
documented weak-supervision and spot-check strategies rather than invented numbers. The
project demonstrates genuine Transformer fine-tuning (intent, urgency), transfer learning
from pretrained checkpoints (sentiment, NER, summarization), and classical NLP techniques
(regex extraction, embedding-based keyphrase extraction) composed into one coherent,
end-to-end system — the `analyze_conversation()` function in Section 16 is the tangible,
demoable artifact of that system.


## 21. CV Bullet Points

Fill in the bracketed numbers with the values printed in the **Final Results Summary**
cell above from your own run (they will differ slightly run-to-run due to normal training
variance, even with a fixed seed, across different hardware/library versions):

- Engineered an end-to-end Transformer-based NLP pipeline for customer-support conversation
  intelligence, fine-tuning DistilBERT for 77-class intent classification on the Banking77
  benchmark and achieving **[XX.X]% accuracy** / **[0.XXX] weighted F1** on a held-out test
  set with zero data leakage.
- Designed and implemented a weak-supervision framework to train an urgency/priority
  classifier in the absence of labeled data — combining a rule-based heuristic with
  DistilBERT fine-tuning and class-weighted loss — validated against a hand-labeled gold
  set with **[XX.X]% agreement** and **[0.XXX] weighted F1**.
- Built a five-model conversational-AI inference pipeline integrating fine-tuned intent
  classification, transfer-learning-based sentiment analysis (RoBERTa), transformer NER
  (BERT), embedding-based keyphrase extraction (KeyBERT), and abstractive dialogue
  summarization (DistilBART/SAMSum) into a single production-style Python function with
  full evaluation, confusion-matrix analysis, and documented error analysis.
